# AulaSense — Versión B: arquitectura orientada a eventos internos

**Universidad Politécnica de Querétaro** · Ingeniería en Datos e Inteligencia Artificial  
**Materia:** Ingeniería en Software · **Actividad 1 — Segundo parcial**  
**Proyecto:** AulaSense — Monitoreo Inteligente de Aulas  
**Responsable:** Ricardo Leonel Ordaz Aguilar · **Grupo:** IDIA211  
**Docente:** Omar Gomez Olvera · **Equipo:** David Israel Atilano Quiroz, Ricardo Leonel Ordaz Aguilar, Javier Yael Olguín Narváez y Daniela Sánchez García  
**Rama:** `ricardo-eventos` · **Ruta:** `02-propuestas/ricardo-eventos/02_ricardo_eventos.ipynb`

> **Base documental:** *AulaSense_Formato_Equipo.pdf*, secciones 1 a 7 (15 de octubre de 2026). Esta versión desarrolla la **alternativa B** asignada a Ricardo en la página 7. Es una propuesta de diseño con demostración local, no una integración ya desplegada con sensores físicos o una base de datos real.

## 1. Nombre, descripción y alcance de la propuesta

**Nombre:** Arquitectura orientada a eventos internos para AulaSense.

AulaSense busca monitorear la ocupación y condiciones ambientales de aulas: presencia, personas estimadas (cuando el sensor lo permite), temperatura, humedad y CO₂ **o** un indicador de calidad del aire. Los nodos IoT transmiten lecturas por Wi-Fi hacia un servidor para almacenamiento, consulta y alertas.

En mi alternativa, la entrada de una lectura válida produce un evento `MedicionValidada`. Un `BusEventos` local lo distribuye a dos componentes independientes: `ManejadorPersistencia` guarda la medición y `ManejadorAlertas` evalúa una regla. La simulación no usa hardware, red, base de datos ni servicio de notificaciones.

**Límite:** se propone una sola aplicación modular con eventos internos síncronos; no se introducen microservicios, modelos avanzados de IA ni automatización de ventilación o iluminación.

## 2. Problema del diseño original y mejora buscada

El documento del equipo presenta **como hipótesis**, no como código implementado y verificado, una clase `SistemaAulaSense` que podría concentrar recepción, validación, procesamiento, persistencia, alertas, consultas, presentación y administración (secciones 2 y 3).

| Riesgo potencial identificado por el equipo | Consecuencia | Respuesta de esta propuesta |
|---|---|---|
| Baja cohesión | La misma clase cambia por motivos distintos | Separar ingesta, distribución, almacenamiento y alertas |
| Dependencias excesivas | Cambiar la persistencia afecta otras funciones | Publicar un evento sin conocer al consumidor |
| Pruebas complicadas | Una regla de alerta depende de otros servicios | Probar consumidores con datos simulados |
| Cambios de gran alcance | Incorporar una nueva acción modifica la clase central | Suscribir otro manejador al bus |

La arquitectura por capas de la sección 4 es la **alternativa A** del equipo. Esta versión B no la reemplaza ni afirma que ya esté implementada: explora una **coordinación distinta mediante publicación/suscripción**.

## 3. Clases propuestas y responsabilidades

| Clase o entidad | Responsabilidad | Método / dato clave |
|---|---|---|
| `Medicion` | Representar una lectura validada y conservar su contexto | aula, dispositivo, fecha, presencia, personas, temperatura, humedad, tipo/valor/unidad de aire |
| `Evento` | Encapsular el hecho ocurrido y su medición | `tipo`, `medicion` |
| `Ingestor` | Recibir y validar datos de entrada; publicar el evento | `recibir(lectura)` |
| `BusEventos` | Registrar suscriptores y distribuir eventos internos | `suscribir(tipo, manejador)`, `publicar(evento)` |
| `ManejadorPersistencia` | Consumir el evento y guardar una medición | `procesar(evento)` |
| `ManejadorAlertas` | Consumir el evento, aplicar reglas configurables y registrar alertas | `procesar(evento)` |

**Relaciones:** `Ingestor → BusEventos`; `BusEventos → ManejadorPersistencia` y `BusEventos → ManejadorAlertas`. Ambos consumidores reciben `MedicionValidada`, pero no se invocan entre sí. Para el prototipo, la persistencia se simula con una lista; una implementación posterior podría utilizar el contrato de `RepositorioMediciones` definido en el diseño general del equipo.

## 4. Diagrama de clases mejorado

```mermaid
classDiagram
    class Medicion {
      +str aula_id
      +str dispositivo_id
      +str fecha_hora
      +bool presencia
      +int personas_estimadas
      +float temperatura_c
      +float humedad_pct
      +str aire_tipo
      +float aire_valor
      +str aire_unidad
    }
    class Evento {
      +str tipo
      +Medicion medicion
    }
    class Ingestor {
      -BusEventos bus
      +recibir(lectura) Medicion
    }
    class BusEventos {
      -dict suscriptores
      +suscribir(tipo, manejador)
      +publicar(evento)
    }
    class ManejadorPersistencia {
      +list registros
      +procesar(evento)
    }
    class ManejadorAlertas {
      +float umbral_co2_ppm
      +list alertas
      +procesar(evento)
    }
    Ingestor --> BusEventos : publica
    Ingestor ..> Medicion : valida y crea
    BusEventos --> Evento : distribuye
    Evento *-- Medicion : contiene
    BusEventos --> ManejadorPersistencia : notifica
    BusEventos --> ManejadorAlertas : notifica
```

**Interpretación:** las flechas indican colaboración o uso, **no herencia**. El nodo IoT es externo al servidor. La persistencia y las alertas son suscriptores separados; se pueden añadir otros sin modificar el `Ingestor`.

### Secuencia de una lectura

```mermaid
sequenceDiagram
    participant N as Nodo IoT (simulado)
    participant I as Ingestor
    participant B as BusEventos
    participant P as ManejadorPersistencia
    participant A as ManejadorAlertas
    N->>I: lectura (payload)
    I->>I: validar campos y unidades
    I->>B: publicar(MedicionValidada)
    B->>P: procesar(evento)
    P-->>B: dato guardado en memoria
    B->>A: procesar(evento)
    A-->>B: alerta si aplica
```

**Decisión de orden para la demostración:** el bus ejecuta primero persistencia y luego alertas. Es una decisión de configuración, no una garantía de entrega durable: si falla un consumidor, el bus registra el error y continúa con los demás.

![Diagrama de clases de AulaSense](diagrama_eventos.png)

## 5. Justificación técnica y beneficios

Esta organización aplica **responsabilidad única**: el ingestor valida, el bus distribuye, la persistencia almacena y el manejador de alertas decide si existe una condición de aviso. Hay **alta cohesión** dentro de cada clase y **menor acoplamiento** porque el productor desconoce qué acciones atienden sus eventos.

| Criterio | Beneficio concreto en AulaSense |
|---|---|
| Mantenimiento | Cambiar el formato de almacenamiento afecta principalmente al manejador de persistencia |
| Reutilización | Otros módulos pueden publicar el mismo tipo de evento y reutilizar suscriptores |
| Pruebas | Se verifican guardado y reglas sin hardware ni PostgreSQL |
| Extensibilidad | Un manejador de auditoría o estadísticas se agrega mediante `suscribir()` |
| Trabajo en equipo | Se pueden implementar y probar manejadores separados con un contrato común de evento |

**Compatibilidad con el informe:** las entidades `Medicion` y `Evento` siguen la idea de la sección 5; el prototipo se mantiene dentro del alcance descrito en la sección 1. La validación y la estimación real de ocupación podrían delegarse más adelante a `ValidadorLectura` y `EstimadorOcupacion`, clases de la alternativa general del equipo.

## 6. Ejemplo funcional en Python con mediciones simuladas

Se simulan tres mediciones ambientales. Dos contienen CO₂ medido en **ppm** y otra usa un **índice genérico de calidad del aire**. No se transforma ese índice en ppm: son variables diferentes, como exige el documento del equipo.

**Regla ilustrativa, no umbral normativo:** generar una alerta cuando una medición de tipo `CO2` en `ppm` supera **1000 ppm**. Para una implementación real, el umbral debe definirse y validarse según el sensor, la normativa y el contexto.

El flujo se ejecuta en memoria para que pueda reproducirse en Jupyter sin instalar bibliotecas externas.

In [1]:
from dataclasses import dataclass
from typing import Callable

@dataclass(frozen=True)
class Medicion:
    aula_id: str
    dispositivo_id: str
    fecha_hora: str
    presencia: bool
    personas_estimadas: int | None
    temperatura_c: float
    humedad_pct: float
    aire_tipo: str
    aire_valor: float
    aire_unidad: str

@dataclass(frozen=True)
class Evento:
    tipo: str
    medicion: Medicion

class BusEventos:
    """Bus interno síncrono; registra errores y continúa con otros consumidores."""
    def __init__(self):
        self.suscriptores: dict[str, list[Callable[[Evento], None]]] = {}
        self.errores: list[str] = []

    def suscribir(self, tipo: str, manejador: Callable[[Evento], None]):
        self.suscriptores.setdefault(tipo, []).append(manejador)

    def publicar(self, evento: Evento):
        for manejador in self.suscriptores.get(evento.tipo, []):
            try:
                manejador(evento)
            except Exception as exc:
                self.errores.append(f"{manejador.__qualname__}: {exc}")

class Ingestor:
    def __init__(self, bus: BusEventos):
        self.bus = bus

    def recibir(self, lectura: dict) -> Medicion:
        requeridos = {
            "aula_id", "dispositivo_id", "fecha_hora", "presencia",
            "personas_estimadas", "temperatura_c", "humedad_pct",
            "aire_tipo", "aire_valor", "aire_unidad"
        }
        faltantes = requeridos - lectura.keys()
        if faltantes:
            raise ValueError(f"Campos faltantes: {sorted(faltantes)}")
        if not all(isinstance(lectura[c], str) and lectura[c].strip()
                   for c in ("aula_id", "dispositivo_id", "fecha_hora")):
            raise ValueError("Aula, dispositivo y fecha deben ser textos no vacíos")
        if type(lectura["presencia"]) is not bool:
            raise ValueError("presencia debe ser booleana")
        personas = lectura["personas_estimadas"]
        if personas is not None and (type(personas) is not int or personas < 0):
            raise ValueError("personas_estimadas debe ser entero >= 0 o None")
        for campo in ("temperatura_c", "humedad_pct", "aire_valor"):
            if type(lectura[campo]) not in (int, float):
                raise ValueError(f"{campo} debe ser numérico")
        if not (0 <= lectura["humedad_pct"] <= 100):
            raise ValueError("humedad_pct fuera del intervalo 0-100")
        if lectura["aire_tipo"] == "CO2" and lectura["aire_unidad"] != "ppm":
            raise ValueError("CO2 debe expresarse en ppm en este prototipo")
        if not isinstance(lectura["aire_tipo"], str) or not isinstance(lectura["aire_unidad"], str):
            raise ValueError("tipo y unidad del aire deben ser textos")
        medicion = Medicion(**lectura)
        self.bus.publicar(Evento("MedicionValidada", medicion))
        return medicion

class ManejadorPersistencia:
    def __init__(self):
        self.registros: list[Medicion] = []

    def procesar(self, evento: Evento):
        self.registros.append(evento.medicion)

class ManejadorAlertas:
    def __init__(self, umbral_co2_ppm: float = 1000):
        self.umbral_co2_ppm = umbral_co2_ppm
        self.alertas: list[str] = []

    def procesar(self, evento: Evento):
        m = evento.medicion
        if m.aire_tipo == "CO2" and m.aire_unidad == "ppm" and m.aire_valor > self.umbral_co2_ppm:
            self.alertas.append(
                f"Alerta en {m.aula_id}: CO2={m.aire_valor} ppm "
                f"(umbral ilustrativo {self.umbral_co2_ppm} ppm)"
            )

print("Clases de la propuesta B definidas correctamente.")

Clases de la propuesta B definidas correctamente.


### 6.1 Configuración del bus y simulación

Primero se suscribe persistencia y después alertas. La primera medición queda bajo el umbral, la segunda lo supera y la tercera representa un indicador de aire distinto de CO₂.

In [2]:
bus = BusEventos()
persistencia = ManejadorPersistencia()
alertas = ManejadorAlertas(umbral_co2_ppm=1000)
ingestor = Ingestor(bus)

bus.suscribir("MedicionValidada", persistencia.procesar)
bus.suscribir("MedicionValidada", alertas.procesar)

lecturas = [
    dict(aula_id="A-101", dispositivo_id="ESP32-01", fecha_hora="2026-10-15T09:00:00",
         presencia=True, personas_estimadas=None, temperatura_c=23.5,
         humedad_pct=49.0, aire_tipo="CO2", aire_valor=850.0, aire_unidad="ppm"),
    dict(aula_id="A-101", dispositivo_id="ESP32-01", fecha_hora="2026-10-15T09:05:00",
         presencia=True, personas_estimadas=None, temperatura_c=24.1,
         humedad_pct=51.0, aire_tipo="CO2", aire_valor=1250.0, aire_unidad="ppm"),
    dict(aula_id="B-202", dispositivo_id="ESP32-02", fecha_hora="2026-10-15T09:10:00",
         presencia=False, personas_estimadas=None, temperatura_c=22.8,
         humedad_pct=46.0, aire_tipo="indice_calidad_aire", aire_valor=78.0, aire_unidad="indice"),
]

for lectura in lecturas:
    m = ingestor.recibir(lectura)
    print(f"Evento MedicionValidada: {m.aula_id}, {m.aire_tipo}={m.aire_valor} {m.aire_unidad}")

print(f"\nMediciones almacenadas en memoria: {len(persistencia.registros)}")
print(f"Alertas generadas: {len(alertas.alertas)}")
for alerta in alertas.alertas:
    print("-", alerta)

Evento MedicionValidada: A-101, CO2=850.0 ppm
Evento MedicionValidada: A-101, CO2=1250.0 ppm
Evento MedicionValidada: B-202, indice_calidad_aire=78.0 indice

Mediciones almacenadas en memoria: 3
Alertas generadas: 1
- Alerta en A-101: CO2=1250.0 ppm (umbral ilustrativo 1000 ppm)


## 7. Pruebas sencillas y resultados esperados

Se comprueba que cada lectura válida se guarda una vez, que solo el CO₂ que supera el umbral genera alerta, que una lectura inválida no se publica y que un fallo de un suscriptor no impide la ejecución de los siguientes. Las pruebas son **locales y simuladas**, no evidencias de hardware real.

In [3]:
# 1. Tres lecturas válidas -> tres registros, una alerta.
assert len(persistencia.registros) == 3
assert len(alertas.alertas) == 1
assert "A-101" in alertas.alertas[0]

# 2. El índice de calidad del aire NO se interpreta como CO2 en ppm.
assert persistencia.registros[2].aire_tipo == "indice_calidad_aire"
assert persistencia.registros[2].aire_unidad == "indice"

# 3. Presencia binaria no implica inventar un número de personas.
assert all(m.personas_estimadas is None for m in persistencia.registros)

# 4. Una lectura inválida se rechaza antes de publicar el evento.
registros_antes = len(persistencia.registros)
try:
    ingestor.recibir({**lecturas[0], "aula_id": ""})
except ValueError:
    pass
else:
    raise AssertionError("Se esperaba rechazar el aula vacía")
assert len(persistencia.registros) == registros_antes

# 5. Un consumidor fallido no bloquea el siguiente (sin reintentos automáticos).
bus_prueba = BusEventos()
recibidos = []
def consumidor_fallido(evento):
    raise RuntimeError("Fallo simulado")
def consumidor_sano(evento):
    recibidos.append(evento.tipo)
bus_prueba.suscribir("MedicionValidada", consumidor_fallido)
bus_prueba.suscribir("MedicionValidada", consumidor_sano)
bus_prueba.publicar(Evento("MedicionValidada", persistencia.registros[0]))
assert len(bus_prueba.errores) == 1
assert recibidos == ["MedicionValidada"]

print("OK: 5 pruebas de diseño y comportamiento completadas.")

OK: 5 pruebas de diseño y comportamiento completadas.


## 8. Limitaciones, riesgos y posibles mejoras

1. **Orden y fallos:** el bus de ejemplo llama a los suscriptores en el orden registrado. Captura excepciones para que un fallo no bloquee a otros, pero **no ofrece reintentos ni garantiza entrega**. En un sistema real habría que definir política de reintentos, trazabilidad e idempotencia.
2. **Persistencia simulada:** una lista en memoria pierde los datos al reiniciar. Para un despliegue real, `ManejadorPersistencia` necesitaría una base de datos y manejo de transacciones.
3. **Procesamiento síncrono:** un consumidor lento retrasa a los demás. Una cola asíncrona sería una evolución posible, no necesaria para este prototipo.
4. **Validación limitada:** el ejemplo valida campos esenciales, pero faltan controles de fechas, autenticidad del dispositivo, rangos específicos por sensor y duplicados.
5. **Alertas ilustrativas:** el umbral de 1000 ppm solo demuestra el flujo. Las reglas reales dependen de sensores y criterios definidos por el equipo.
6. **Ocupación:** la presencia booleana no permite deducir automáticamente cuántas personas hay; por ello `personas_estimadas` se mantiene en `None` cuando no hay método de conteo.

Estas limitaciones responden al requisito del documento de equipo: la versión B debe resolver explícitamente **orden y fallos**, sin introducir servicios distribuidos.

## 9. Conclusión

La propuesta B cambia la forma de coordinar el servidor AulaSense: en lugar de que una clase central invoque directamente todos los procesos, `Ingestor` publica `MedicionValidada` y `BusEventos` distribuye ese hecho a componentes especializados. El ejemplo demuestra que una lectura puede almacenarse y, si cumple una regla, generar una alerta, manteniendo independientes ambos comportamientos.

Frente al diseño original **hipotético** del equipo, esta alternativa facilita mantenimiento, reutilización, pruebas y futuras funciones. Su costo es que requiere definir contratos de eventos y gestionar orden, fallos y duplicados. Para el alcance académico actual resulta razonable mantener un bus interno simple, sin infraestructura distribuida.

**Fuente base:** *AulaSense_Formato_Equipo.pdf*, secciones 1 (alcance), 2–3 (diseño original supuesto), 4–6 (clases y principios de mejora) y 7 (versión B asignada a Ricardo).